# Magneto-quasistatics

An electric current generates a magnetic field. In the magneto-quasistatic regime, wave propagation is neglected but induction and material effects remain.

## Vector potential and its two-dimensional reduction

Writing the magnetic flux density as $\vec B=\operatorname{curl}\vec A$ leads to the magnetic vector-potential equation

$$
\kappa \vec A+\operatorname{curl}\bigl(\nu\operatorname{curl}\vec A\bigr)=\vec J_s,
$$

where $\nu=\mu^{-1}$ is the magnetic reluctivity and $\vec J_s$ is the prescribed source-current density. The effective reaction coefficient $\kappa$ arises from electrical conductivity in a transient or time-harmonic eddy-current model.

For a two-dimensional configuration with current perpendicular to the drawing plane, the vector potential has only one component, $\vec A=(0,0,A_z(x,y))$. The model becomes the scalar elliptic equation

$$
\kappa A_z-\operatorname{div}(\nu\nabla A_z)=J_{s,z}.
$$

In this experiment we set $\kappa=0$, so the problem is magnetostatic. The two circles represent conductor cross-sections carrying equal currents in opposite directions. The in-plane magnetic flux density is recovered from

$$\vec B=(\partial_yA_z,-\partial_xA_z).$$

In [ ]:
from netgen.occ import OCCGeometry, Rectangle, WorkPlane, Glue
from ngsolve import (
    Mesh, H1, GridFunction, BilinearForm, LinearForm, CF,
    Grad, dx
)
from ngsolve.webgui import Draw

outer = Rectangle(3, 2).Face().Move((-1.5, -1, 0))
outer.edges.name = "outer"
coil_plus = WorkPlane().MoveTo(-0.5, 0).Circle(0.22).Face()
coil_minus = WorkPlane().MoveTo(0.5, 0).Circle(0.22).Face()
coil_plus.faces.name = "coil_plus"
coil_minus.faces.name = "coil_minus"
air = outer - coil_plus - coil_minus
air.faces.name = "air"

geometry = Glue([air, coil_plus, coil_minus])
mesh = Mesh(OCCGeometry(geometry, dim=2).GenerateMesh(maxh=0.12)).Curve(2)
Draw(mesh);

In [ ]:
# Numerical solver
fes = H1(mesh, order=2, dirichlet="outer")
u, v = fes.TnT()

# Nondimensional magnetic reluctivity nu = 1/mu
magnetic_reluctivity = mesh.MaterialCF({
    "air": 1.0, "coil_plus": 0.2, "coil_minus": 0.2
})
# Out-of-plane source-current density J_s,z
source_current_density_z = mesh.MaterialCF({
    "air": 0.0, "coil_plus": 1.0, "coil_minus": -1.0
})

A = BilinearForm(fes)
A += magnetic_reluctivity * Grad(u) * Grad(v) * dx
f = LinearForm(fes)
f += source_current_density_z * v * dx
A.Assemble()
f.Assemble()

vector_potential_z = GridFunction(fes)
vector_potential_z.vec.data = A.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec
magnetic_flux_density = CF((Grad(vector_potential_z)[1], -Grad(vector_potential_z)[0]))

In [ ]:
Draw(vector_potential_z, mesh, "vector potential A_z")
Draw(magnetic_flux_density, mesh, "magnetic flux density B", vectors={"grid_size": 36});

## Observe

- How does reversing one current change the symmetry of the field?
- Why do the magnetic-flux-density vectors follow level curves of $A_z$?
- What does the outer condition $A_z=0$ approximate physically?

[← Incompressible flow](05_incompressible_flow.ipynb) · [Lecture overview](index.ipynb)